# K2 - G7 re-run: what the hub does to local structure  (C.13.10)

Regenerates the per-pair gains that died with the VM. **Writes `g7_rebuilt.npz`, which K3 reads.**

Gates on the healthy/degenerate structure reproducing rather than on exact agreement - the rebuilt protocol is known to differ.

## Storage

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

## Experiment

In [ ]:
# ==========================================================
# K2 — G7 re-run: what the hub does to local structure (report C.13.10).
# Produces g7_rebuilt.npz, which K3 consumes. Needs G0 to have passed.
#
# WHY THIS EXISTS. The original G7 per-pair gains died with the Colab VM
# like the rest of the G-series artifacts. K3 cannot compute an interval
# over six numbers it does not have, so they have to be remeasured.
#
# WHAT G7 MEASURES. The science says what encoders share is LOCAL; the
# engineering is a GLOBAL linear projection. G7 makes them meet: for each
# pair, does mapping both spaces through the hub increase or decrease
# their neighbourhood agreement?
#
#   gain = kNN overlap AFTER the hub  -  kNN overlap BEFORE
#
# The published answer, which this must reproduce closely enough to
# trust: aggregate +0.111, but that splits in two once pairs are
# stratified by the collapse diagnostic. On HEALTHY pairs (worst
# pair-cosine < 0.30) the hub is NEUTRAL, +0.010 over n=6 with two
# negative. On pairs including a DEGENERATE space it gains +0.151, with
# Spearman +0.82 between a pair's worst pair-cosine and its gain - the
# C.11 isotropy rescue for the fourth independent time, not hub alignment.
#
# THE FLAG IS PAIR-COSINE, NOT EFFECTIVE RANK. An earlier version of G7
# used effective rank divided by ambient width, which misclassified SBERT
# - the healthiest text space in the project - as degenerate, and reported
# the OPPOSITE verdict. Pair-cosine is width-independent and does not have
# that failure mode. This is recorded in C.13.10 and is not re-litigated
# here; the cell uses pair-cosine.
#
# WHAT TO CHECK BEFORE TRUSTING THE OUTPUT. Three published numbers are
# reprinted alongside the new ones: mean hub preservation 0.594, GPT-2
# worst at 0.257, and the healthy/degenerate split above. These come from
# the ORIGINAL hub. This runs on the rebuilt one, so exact agreement is
# not expected - the rebuilt protocol already runs measurably different in
# G4 (-7.8 points). What matters is that the STRUCTURE reproduces: healthy
# near zero, degenerate clearly positive, Spearman clearly positive, and
# the same encoders on the same side of the flag.
# ==========================================================
import os
import numpy as np
from pathlib import Path
from itertools import combinations

DATA_DIR = Path(os.environ["DATA_DIR"])
HUB_NPZ = DATA_DIR / "hub_rebuilt.npz"
HUB_WIDTH = 512          # the operating point
K = 10                   # neighbourhood size, as in C.13.10
N_SAMPLE = 2000          # rows used for the kNN computation
COLLAPSE_FLAG = 0.30     # pair-cosine above this is degenerate
SEED = 0

PUB_PRESERVE_MEAN = 0.594
PUB_PRESERVE_GPT2 = 0.257
PUB_HEALTHY = 0.010
PUB_DEGENERATE = 0.151
PUB_SPEARMAN = 0.82

hub = np.load(HUB_NPZ, allow_pickle=True)
names = [str(n) for n in hub["encoder_names"]]
N_TR = int(hub["n_train"])
X = {n: hub[f"raw_{n}"].astype(np.float64) for n in names}
H_all = hub["H_all"][:, :HUB_WIDTH]
print(f"{len(names)} encoders, hub at {HUB_WIDTH}-d: {', '.join(names)}")

rng = np.random.default_rng(SEED)
idx = rng.choice(len(H_all), min(N_SAMPLE, len(H_all)), replace=False)


def l2(A):
    return A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-8)


def neighbours(A, k=K):
    a = l2(A[idx])
    return np.argsort(-(a @ a.T), axis=1)[:, 1:k + 1]


def overlap(na, nb, k=K):
    return float(np.mean([len(set(x) & set(y)) / k for x, y in zip(na, nb)]))

In [ ]:
# ---------- per-space hub coordinates, one linear map each ----------
from sklearn.linear_model import Ridge

H_of = {}
for n in names:
    W = Ridge(alpha=1.0).fit(X[n][:N_TR], H_all[:N_TR]).coef_.T
    H_of[n] = X[n] @ W

In [ ]:
# ---------- collapse diagnostic, per space ----------
pair_cos_space = {}
for n in names:
    a = l2(X[n][idx])
    S = a @ a.T
    pair_cos_space[n] = float(S[np.triu_indices(len(idx), 1)].mean())

print(f"\nmean pair-cosine per space (flag > {COLLAPSE_FLAG}):")
for n in sorted(names, key=lambda z: -pair_cos_space[z]):
    tag = "  DEGENERATE" if pair_cos_space[n] > COLLAPSE_FLAG else ""
    print(f"  {n:12s} {pair_cos_space[n]:+.3f}{tag}")

In [ ]:
# ---------- native-neighbourhood preservation ----------
print(f"\nhub preservation at k={K} (published mean {PUB_PRESERVE_MEAN}, "
      f"GPT-2 {PUB_PRESERVE_GPT2}):")
preserve = {}
for n in names:
    preserve[n] = overlap(neighbours(X[n]), neighbours(H_of[n]))
    print(f"  {n:12s} {preserve[n]:.3f}")
print(f"  {'mean':12s} {np.mean(list(preserve.values())):.3f}")

In [ ]:
# ---------- per-pair gains ----------
nb_before = {n: neighbours(X[n]) for n in names}
nb_after = {n: neighbours(H_of[n]) for n in names}

pair_names, gains, pcos = [], [], []
for a, b in combinations(sorted(names), 2):
    before = overlap(nb_before[a], nb_before[b])
    after = overlap(nb_after[a], nb_after[b])
    pair_names.append(f"{a} - {b}")
    gains.append(after - before)
    pcos.append(max(pair_cos_space[a], pair_cos_space[b]))

gains = np.array(gains)
pcos = np.array(pcos)
healthy = pcos < COLLAPSE_FLAG

print("\n" + "=" * 68)
print(f"{'pair':<30}{'before':>9}{'after':>9}{'gain':>9}{'worst pcos':>11}")
print("=" * 68)
for i in np.argsort(-gains):
    a, b = pair_names[i].split(" - ")
    bf = overlap(nb_before[a], nb_before[b])
    af = overlap(nb_after[a], nb_after[b])
    tag = "" if healthy[i] else "  D"
    print(f"{pair_names[i]:<30}{bf:>9.3f}{af:>9.3f}{gains[i]:>+9.3f}"
          f"{pcos[i]:>11.3f}{tag}")

In [ ]:
# ---------- the stratified reading ----------
def spearman(x, y):
    rx = np.argsort(np.argsort(x)).astype(float)
    ry = np.argsort(np.argsort(y)).astype(float)
    rx -= rx.mean(); ry -= ry.mean()
    return float((rx @ ry) / np.sqrt((rx @ rx) * (ry @ ry)))


rho = spearman(pcos, gains)
print("\n" + "=" * 68)
print(f"  aggregate over all {len(gains)} pairs     "
      f"{gains.mean():+.3f}")
print(f"  HEALTHY pairs (n={healthy.sum()})            "
      f"{gains[healthy].mean():+.3f}   "
      f"published {PUB_HEALTHY:+.3f}")
print(f"    negative among them          "
      f"{int((gains[healthy] < 0).sum())} of {int(healthy.sum())}")
print(f"  DEGENERATE pairs (n={(~healthy).sum()})         "
      f"{gains[~healthy].mean():+.3f}   "
      f"published {PUB_DEGENERATE:+.3f}")
print(f"  Spearman(worst pcos, gain)     {rho:+.3f}   "
      f"published {PUB_SPEARMAN:+.3f}")

print("\n" + "=" * 68)
ok_struct = (abs(gains[healthy].mean()) < 0.05
             and gains[~healthy].mean() > 0.05
             and rho > 0.4)
if ok_struct:
    print("STRUCTURE REPRODUCED. Healthy near zero, degenerate clearly")
    print("positive, Spearman positive. The aggregate is not one effect: it")
    print("is the C.11 isotropy rescue showing up in the pairs that include")
    print("a degenerate space, and near-nothing everywhere else. The large")
    print("gains must NOT be averaged into a headline.")
else:
    print("STRUCTURE DID NOT REPRODUCE. The rebuilt hub gives a different")
    print("stratification from the published one. Do not feed this into K3:")
    print("an interval computed over gains that disagree with C.13.10 would")
    print("bound the wrong quantity. Report C.13.10 as it stands.")

if healthy.sum() != 6:
    print(f"\n  NOTE: {healthy.sum()} healthy pairs, not the 6 in the report.")
    print("  K3's cluster bootstrap assumes the pairs are all combinations")
    print("  of a small encoder set - check that still holds before running it.")

out = DATA_DIR / "g7_rebuilt.npz"
np.savez_compressed(out, pair_names=np.array(pair_names), gain=gains,
                    pair_cos=pcos,
                    preserve=np.array([preserve[n] for n in names]),
                    encoder_names=np.array(names), spearman=rho)
print(f"\nwrote {out.name} - K3 reads this")